In [1]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchsummary import summary
from PIL import Image

import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn


In [2]:
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted successfully!")
else:
    print("Drive already mounted.")

Drive already mounted.


### Clone git and load modules

In [3]:
!git clone https://github.com/gimoonnam/vgg16_practice.git

fatal: destination path 'vgg16_practice' already exists and is not an empty directory.


In [4]:
repo_path = '/content/vgg16_practice'
if repo_path not in sys.path:
  sys.path.insert(0, repo_path)

from load_data import CatandDogDataLoader
from vgg16_model import VGG16

### Load data and Save dataset as ubyte format


In [5]:
data_path = r'/content/drive/My Drive/Data for Colab Training'
train_data_path = os.path.join(data_path, "cat-and-dog")
test_data_path  = os.path.join(data_path, "cat-and-dog")

# # load data
# ds_train = CatDogDataLoadandSave(data_dir=train_data_path)
# ds_test = CatDogDataLoadandSave(data_dir=test_data_path)

# save them as ubyte format
# output_directory = os.path.join(data_path, "cat-and-dog/ubyte_format")
# ds_train.save_as_ubyte(output_dir=output_directory, img_size=(224, 224), prefix="catdog_train")
# ds_test.save_as_ubyte(output_dir=output_directory, img_size=(224, 224), prefix="catdog_test")

### create torch data loader

In [6]:
dataset_train = CatandDogDataLoader(raw_folder=train_data_path, train=True)
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=64, shuffle=True)

Loaded 8005 images of shape 224x224x3
Loaded 2049 labels


### Build VGG16 architecture

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VGG16(3, 2).to(device)

device

device(type='cuda')

In [8]:
from torchsummary import summary
summary(model, (3, 224, 224))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            N_conv-6         [-1, 64, 112, 112]               0
            Conv2d-7        [-1, 128, 112, 112]          73,856
              ReLU-8        [-1, 128, 112, 112]               0
            Conv2d-9        [-1, 128, 112, 112]         147,584
             ReLU-10        [-1, 128, 112, 112]               0
        MaxPool2d-11          [-1, 128, 56, 56]               0
           N_conv-12          [-1, 128, 56, 56]               0
           Conv2d-13          [-1, 256, 56, 56]         295,168
             ReLU-14          [-1, 256,

In [9]:
learning_rate = 1e-4
num_epochs = 20

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [11]:
model.train()

for epoch in range(num_epochs):
    ProgressBar = tqdm(enumerate(train_loader), total=len(train_loader))

    for batch_idx, (inputs, labels) in ProgressBar:

        # Ensure labels are torch.long before moving to device for CrossEntropyLoss
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.long()
        # print(inputs.dtype, labels.dtype)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        #Update Progress bar
        ProgressBar.set_description(f'Epoch [{epoch+1}]')
        ProgressBar.set_postfix(loss=loss.item())

  0%|          | 0/126 [00:00<?, ?it/s]


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
